# 4 · Repeatable checks and shareable reports

**The story.** Every result in the previous notebooks rested on checks run by hand, in order. That order is repeatable — and repeatable steps belong in **the model file itself**, so that anyone (or any tool) can rerun them. Dyno lets the file declare its own workflow with `@run:` directives, execute it with `model.run()`, and render everything — equations, diagnostics, figures — into a single human-readable **report**: a receipt of the model's health.

Note: `model.run()` and the `@run:` machinery are currently experimental, so keep the model file and its directives together when sharing an analysis.

In [1]:
import numpy as np
import pandas as pd
from dyno import DynoModel
from dyno.report import dsge_report

pd.set_option("display.precision", 4)

## 1. The file declares its own workflow

The bottom of [models/rbc.dyno](models/rbc.dyno) now declares the whole recipe we ran by hand in notebook 1:

```text
@run: check
@run: solve
@run: simul: {T: 40, mode: irf, units: percent}
@run: plot: {variables: [a, y, c, i]}
```

`model.run()` executes the steps in order and returns a `RunResults` object that remembers each output — residuals, eigenvalues, solution, simulation and the plot — plus any warnings and errors.

In [2]:
model = DynoModel("models/rbc.dyno")
results = model.run()
print("Pipeline finished with", len(results.errors), "errors and", len(results.warnings), "warnings")
assert not results.errors

Pipeline finished with 0 errors and 0 warnings


### Inspect the diagnostics

A successful pipeline stores the model, solution, Blanchard–Kahn result and residuals. A compact diagnostic table is easier to scan than the full report:

In [3]:
diagnostics = pd.Series({
    "BK conditions hold": bool(results.bk_check),
    "Maximum absolute residual": float(np.max(np.abs(results.residuals))),
    "Solution computed": results.solution is not None,
    "Simulation computed": results.simulation is not None,
}, name="Result")
display(diagnostics.to_frame())
assert diagnostics["BK conditions hold"]
assert diagnostics["Maximum absolute residual"] < 1e-8

,Result
BK conditions hold,True
Maximum absolute residual,0.0
Solution computed,True
Simulation computed,True


### The pipeline drew the plot, too

Because the file asked for a simulation *and* a plot, `run()` already produced the same impulse-response chart as notebook 1 — re-derived from the file, not copied from the notebook:

In [4]:
results.simulation.plot(units="percent", variables=["a", "y", "c", "i"])

alt.Chart(...)

## 2. Render the shareable report

`dsge_report(filename=...)` loads the same file, reruns its directives, and returns the rich report. Inspect the report for errors and warnings before sharing it:

In [5]:
report = dsge_report(filename="models/rbc.dyno")
assert not report.errors
report.display()

The same report exists as plain text (with optional ASCII plots) — same content, no browser — useful in terminals, logs and issue threads:

In [6]:
print("\n".join(report.to_text(graphs=False).splitlines()[:38]))

RunResults
Model
-----
name: rbc
filename: models/rbc.dyno
deterministic: False
symbols: variables=9, endogenous=8, exogenous=1, parameters=7
endogenous: r, n, k, y, i, c, w, a
exogenous: epsilon
parameters: beta, delta, nss, eta, alpha, rho, chi

Checks
------
Residuals: computed (n=8, max|r|=8.882e-16, nonzero@1e-06=0)
Eigenvalues: computed (n=16, min|lambda|=0.000e+00, max|lambda|=inf, |lambda|>1: 8, |lambda|≈1: 0)
Blanchard-Kahn conditions: met

Outputs
-------
Solution: computed
  decision rule matrices: X(8, 8), Y(8, 1), Σ(1, 1)
Simulation: computed (IRFs, shocks=1, horizon=41)
  shocks: epsilon
Figure: available (Chart)
Moments: not available

Diagnostics
-----------
Warnings: none
Errors: none

Timing
------
elapsed: 0.016s


## 3. Work in Dyno Lab

The Pixi environment includes `jupyterlab-dyno`. Run `pixi run lab` from the repository root and open a `.dyno` or `.mod` file to get the editor, live model preview, and error highlighting. The notebooks remain the right tool when the analysis continues in Python.

**Takeaway.** Keep the model text as the reproducible specification. Declare the checks *inside* the file, rerun them with one call, and hand the report to whoever needs to trust the results.